# 🖼️ NumPy Through Image Processing
### An interactive, visual introduction to NumPy for students

> **How to use this notebook:** Run each cell top-to-bottom. Every NumPy concept is introduced only when the image processing task *needs* it. You'll build a mini photo editor by the end.

---
**What you'll learn (and what you'll build along the way):**

| Module | NumPy Concept | You'll build… |
|--------|--------------|---------------|
| 1 | Arrays, shape, dtype | Load & inspect an image |
| 2 | Indexing & slicing | Crop, flip, split RGB channels |
| 3 | Broadcasting & vectorised ops | Brightness, contrast, blur |
| 4 | Aggregation & statistics | Histogram equalisation |
| 5 | Boolean masks & fancy indexing | Green-screen + watermark |

---


## ⚙️ Setup — run this first

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import urllib.request, io, warnings
warnings.filterwarnings('ignore')

# ── helper: show one or more images side-by-side ─────────────────────────────
def show(*imgs, titles=None, cmap=None):
    n = len(imgs)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1: axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        cm = cmap if cmap else ('gray' if img.ndim == 2 else None)
        ax.imshow(img, cmap=cm, vmin=0, vmax=255)
        ax.axis('off')
        if titles: ax.set_title(titles[i], fontsize=11)
    plt.tight_layout()
    plt.show()

# ── download a sample photo (CC0 – Unsplash) ─────────────────────────────────
URL = "https://images.unsplash.com/photo-1518791841217-8f162f1912da?w=640&q=80"
try:
    with urllib.request.urlopen(URL, timeout=5) as r:
        raw = Image.open(io.BytesIO(r.read())).convert("RGB").resize((640, 426))
        raw.save("/tmp/sample.jpg")
    print("✅  Photo downloaded.")
except Exception:
    # fallback: synthetic gradient image
    arr = np.zeros((426, 640, 3), dtype=np.uint8)
    for c, v in enumerate([180, 120, 60]):
        arr[:, :, c] = np.linspace(0, v, 640, dtype=np.uint8)
    Image.fromarray(arr).save("/tmp/sample.jpg")
    print("✅  Using synthetic fallback image.")


---
## Module 1 — Images are just arrays

> **The big idea:** A digital image is nothing more than a grid of numbers. NumPy is the fastest way to work with grids of numbers in Python.

Every pixel has three numbers — Red, Green, Blue — each from 0 to 255.

```
image shape:  (height, width, channels)
                  ↑        ↑       ↑
               rows    columns   RGB = 3
```


In [ ]:
# Load the image with PIL, then convert to a NumPy array
img = np.array(Image.open("/tmp/sample.jpg"))

print("Type    :", type(img))
print("Shape   :", img.shape,  "  ← (height, width, channels)")
print("Dtype   :", img.dtype,  "  ← uint8 means 0–255 per channel")
print("Min px  :", img.min(),  "  Max px:", img.max())
print("Memory  :", img.nbytes // 1024, "KB")


In [ ]:
# Show the image
show(img, titles=["Our sample image"])


### 🔬 Inspect individual pixels

`img[row, col]` gives you the RGB triplet at that pixel.


In [ ]:
# Top-left pixel
print("Top-left pixel  :", img[0, 0])

# Centre pixel
h, w = img.shape[:2]
print("Centre pixel    :", img[h//2, w//2])

# A small patch (3×3 pixels) from the top-left
print("\n3×3 patch (top-left):\n", img[:3, :3])


### 🧪 Exercise 1
Change the numbers below and re-run. What colour does each combination make?


In [ ]:
# Create a tiny 1-pixel "image" manually
pixel = np.array([[[255, 0, 0]]], dtype=np.uint8)   # ← try [0,255,0] or [0,0,255]
show(pixel, titles=[f"RGB = {pixel[0,0].tolist()}"])


---
## Module 2 — Slicing & indexing

> **The big idea:** NumPy slices work on every dimension at once. `img[y1:y2, x1:x2]` crops the image. `img[:, ::-1]` flips it. It's the same slice syntax you know from Python lists — but in 2D.

```
img[row_start : row_stop, col_start : col_stop]
```


In [ ]:
# ── Crop a region ────────────────────────────────────────────────────────────
crop = img[60:220, 200:440]   # rows 60–220, columns 200–440
show(img, crop, titles=["Original", "Cropped region"])
print("Original shape:", img.shape)
print("Crop shape    :", crop.shape)


In [ ]:
# ── Flip horizontally (mirror) ───────────────────────────────────────────────
# ::-1 means "all columns, backwards"
flipped = img[:, ::-1]
show(img, flipped, titles=["Original", "Horizontally flipped"])


In [ ]:
# ── Flip vertically ──────────────────────────────────────────────────────────
flipped_v = img[::-1, :]
show(img, flipped_v, titles=["Original", "Vertically flipped"])


### Split the image into R, G, B channels

`img[:, :, 0]` means: *all rows, all columns, channel 0 (Red)*.


In [ ]:
R = img[:, :, 0]   # Red channel
G = img[:, :, 1]   # Green channel
B = img[:, :, 2]   # Blue channel

show(img, R, G, B,
     titles=["Original", "Red channel", "Green channel", "Blue channel"])

print("Bright reds in image  → R channel will look bright there")
print("Channel shape:", R.shape, "← 2D, no colour dimension")


### 🧪 Exercise 2
What do you think `img[::2, ::2]` does? Run it and check your answer.


In [ ]:
# Your experiment here ↓
result = img[::2, ::2]
show(img, result, titles=["Original", "img[::2, ::2]"])
print("Result shape:", result.shape)


---
## Module 3 — Broadcasting, vectorised ops & filters

> **The big idea:** NumPy applies operations to *every element at once* — no Python `for` loop needed. Multiplying an image array by 1.5 brightens every single pixel in one line.
>
> **Broadcasting** is what happens when NumPy "stretches" a smaller array to match a larger one automatically.

```
img * 1.5   →   scalar 1.5 broadcasts across shape (426, 640, 3)
                no loop, runs in C, ultra-fast
```


In [ ]:
# ── Brightness ───────────────────────────────────────────────────────────────
# Multiply every pixel value by a factor
# np.clip keeps values inside 0–255, then we cast back to uint8
brighter = np.clip(img.astype(np.float32) * 1.6, 0, 255).astype(np.uint8)
darker   = np.clip(img.astype(np.float32) * 0.4, 0, 255).astype(np.uint8)

show(darker, img, brighter,
     titles=["Darker (×0.4)", "Original", "Brighter (×1.6)"])


### Why `float32` then back to `uint8`?

`uint8` can only hold 0–255. If a pixel is 200 and you multiply by 1.6 → 320, which *wraps* to 64 in uint8 (wrong!). We:
1. Cast to float so maths works correctly
2. Clip to 0–255
3. Cast back to uint8 for display


In [ ]:
# Demonstrate the wrap-around problem
bad  = (img * 2).astype(np.uint8)          # ← wrap-around bug!
good = np.clip(img.astype(np.float32) * 2, 0, 255).astype(np.uint8)
show(bad, good, titles=["❌ Wrap-around bug", "✅ np.clip fix"])


### Adjusting contrast

Contrast = how spread out the pixel values are.
```
contrast = (img - 128) * factor + 128
```


In [ ]:
f = img.astype(np.float32)
low_contrast  = np.clip((f - 128) * 0.5 + 128, 0, 255).astype(np.uint8)
high_contrast = np.clip((f - 128) * 2.0 + 128, 0, 255).astype(np.uint8)

show(low_contrast, img, high_contrast,
     titles=["Low contrast (×0.5)", "Original", "High contrast (×2)"])


### Build a blur filter from scratch — manual convolution

A blur works by replacing each pixel with the **average of its neighbours**.
We'll implement a simple box blur using a sliding window — no external libraries.

```
kernel = [[1,1,1],        each pixel gets replaced by
           [1,1,1],   →    the average of itself and
           [1,1,1]] / 9    the 8 pixels around it
```


In [ ]:
def box_blur(image, radius=2):
    """Blur by averaging a (2r+1)×(2r+1) neighbourhood. Pure NumPy."""
    f = image.astype(np.float32)
    out = np.zeros_like(f)
    r = radius
    h, w = f.shape[:2]
    # For each pixel (i,j) average the patch around it
    for i in range(r, h - r):
        for j in range(r, w - r):
            out[i, j] = f[i-r:i+r+1, j-r:j+r+1].mean(axis=(0, 1))
    return np.clip(out, 0, 255).astype(np.uint8)

# Use on a thumbnail so the loop is fast (teach the concept, then scipy for prod)
thumb = img[100:200, 200:340]
blurred = box_blur(thumb, radius=2)
show(thumb, blurred, titles=["Original patch", "Box blur (r=2)"])
print("Each pixel = mean of a 5×5 patch of its neighbours")


### 🧪 Exercise 3
Try making the image **greyscale** using broadcasting:
```python
grey = img @ np.array([0.299, 0.587, 0.114])
```
`@` is the matrix multiply operator. What shape do you expect?


In [ ]:
# Convert to greyscale using luminance weights
grey = (img @ np.array([0.299, 0.587, 0.114])).astype(np.uint8)
show(img, grey, titles=["Original", "Greyscale"])
print("Greyscale shape:", grey.shape, "← 2D, one value per pixel")


---
## Module 4 — Aggregation & statistics → histogram equalisation

> **The big idea:** `np.mean`, `np.std`, `np.histogram`, `np.cumsum` let you *measure* what's in an array. Here we use them to automatically improve a dark photo.

**What is histogram equalisation?**
It spreads pixel values more evenly across 0–255 so that dark images become clearer.


In [ ]:
# ── Create a deliberately dark version of our image ─────────────────────────
dark = np.clip(img.astype(np.float32) * 0.35, 0, 255).astype(np.uint8)

# ── Plot pixel value histogram ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, image, label in zip(axes, [img, dark], ["Original", "Dark version"]):
    grey_vals = (image @ [0.299, 0.587, 0.114]).astype(np.uint8).ravel()
    ax.hist(grey_vals, bins=64, color='steelblue', alpha=0.8)
    ax.set_title(label); ax.set_xlabel("Pixel value (0–255)"); ax.set_ylabel("Count")
    ax.axvline(grey_vals.mean(), color='red', linestyle='--', label=f'mean={grey_vals.mean():.0f}')
    ax.legend()
plt.tight_layout(); plt.show()

print("Dark image mean:", dark.mean().round(1), "← clustered near 0")
print("Original mean  :", img.mean().round(1))


In [ ]:
def histogram_equalise(image):
    """Equalise each channel independently using cumulative histogram."""
    out = np.zeros_like(image)
    for c in range(3):                           # R, G, B
        channel = image[:, :, c]
        # 1. count how often each value (0–255) appears
        hist, _ = np.histogram(channel.ravel(), bins=256, range=(0, 256))
        # 2. cumulative sum → tells us how many pixels are ≤ each value
        cdf = hist.cumsum()
        # 3. normalise cdf to 0–255
        cdf_min = cdf[cdf > 0].min()
        n_pixels = channel.size
        lut = np.round((cdf - cdf_min) / (n_pixels - cdf_min) * 255).astype(np.uint8)
        # 4. apply lookup table — fancy indexing!
        out[:, :, c] = lut[channel]
    return out

equalised = histogram_equalise(dark)
show(dark, equalised, titles=["Dark image", "After histogram equalisation"])


In [ ]:
# Compare histograms before vs after
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, image, label in zip(axes, [dark, equalised], ["Dark", "Equalised"]):
    grey = (image @ [0.299, 0.587, 0.114]).astype(np.uint8).ravel()
    ax.hist(grey, bins=64, color='steelblue', alpha=0.8)
    ax.set_title(label); ax.set_xlabel("Pixel value"); ax.set_xlim(0, 255)
plt.tight_layout(); plt.show()
print("After equalisation the histogram spreads across the full 0–255 range.")


### 🧪 Exercise 4
What does `np.percentile(img, 95)` return? What does it tell you about the image?


In [ ]:
for pct in [5, 25, 50, 75, 95]:
    print(f"  {pct}th percentile: {np.percentile(img, pct):.1f}")


---
## Module 5 — Boolean masks, fancy indexing & composition

> **The big idea:** You can use a boolean array (True/False) as an index into another array. NumPy returns or modifies only the elements where the mask is True.

This is *the* tool for selecting pixels by colour — the core of green-screen removal.


In [ ]:
# ── Create a synthetic green-screen image ────────────────────────────────────
# (A solid green background with a white rectangle = our "subject")
gs = np.zeros((300, 400, 3), dtype=np.uint8)
gs[:, :] = [0, 200, 0]        # fill everything green
gs[80:220, 120:280] = [240, 230, 210]   # white-ish subject box

# Draw a simple "face" on the subject
gs[110:160, 150:200] = [255, 200, 150]  # face colour
gs[115:125, 157:167] = [80, 60, 40]    # left eye
gs[115:125, 183:193] = [80, 60, 40]    # right eye
gs[145:152, 162:188] = [180, 80, 80]   # mouth

show(gs, titles=["Synthetic green-screen image"])


In [ ]:
# ── Build the green-screen mask ──────────────────────────────────────────────
# A pixel is "green background" if:  G channel high  AND  R+B channels low
mask = (gs[:, :, 1] > 150) & (gs[:, :, 0] < 100) & (gs[:, :, 2] < 100)

print("mask dtype:", mask.dtype, " ← boolean!")
print("mask shape:", mask.shape)
print("Green pixels found:", mask.sum())

# Visualise the mask
show(gs, mask.astype(np.uint8) * 255,
     titles=["Original", "Green-screen mask (white = green pixels)"])


In [ ]:
# ── Replace background with a gradient sky ───────────────────────────────────
result = gs.copy()

# Create a sky gradient (blue at top → white at bottom)
sky_row = np.linspace([100, 160, 255], [220, 235, 255], 300).astype(np.uint8)
sky = np.broadcast_to(sky_row[:, np.newaxis, :], gs.shape).copy()

# np.where: use sky where mask is True, keep original elsewhere
result = np.where(mask[:, :, np.newaxis], sky, gs)

show(gs, result, titles=["Green screen", "Background replaced"])


### Add a watermark using alpha blending

Alpha blending formula:
```
output = (1 - α) × original + α × watermark
```
`α = 0` → fully original, `α = 1` → fully watermark.


In [ ]:
# ── Add a semi-transparent watermark text block ──────────────────────────────
watermarked = result.copy().astype(np.float32)
alpha = 0.35   # opacity of watermark

# Create a white rectangle for the watermark band
h, w = watermarked.shape[:2]
wm_band = np.ones((30, w, 3), dtype=np.float32) * 255

# Blend the bottom 30 rows
watermarked[h-30:h] = (1 - alpha) * watermarked[h-30:h] + alpha * wm_band
watermarked = watermarked.astype(np.uint8)

show(result, watermarked, titles=["Background replaced", "With watermark band"])
print(f"Watermark alpha = {alpha} → {int(alpha*100)}% watermark, {int((1-alpha)*100)}% original")


---
## 🎓 Capstone — your mini photo editor

You've now built every tool. Chain them together below:
1. Load the sample image  
2. Crop to a region of interest  
3. Adjust brightness  
4. Equalise the histogram  
5. Overlay a watermark  

Modify the parameters and make it your own!


In [ ]:
# ── Full pipeline ─────────────────────────────────────────────────────────────

# 1. Load
original = np.array(Image.open("/tmp/sample.jpg"))

# 2. Crop  (change these!)
cropped = original[50:376, 80:560]

# 3. Darken then equalise  (simulates fixing a bad exposure)
darkened   = np.clip(cropped.astype(np.float32) * 0.4, 0, 255).astype(np.uint8)
eq         = histogram_equalise(darkened)

# 4. Boost contrast slightly
final = np.clip((eq.astype(np.float32) - 128) * 1.3 + 128, 0, 255).astype(np.uint8)

# 5. Watermark band
f32 = final.astype(np.float32)
h, w = f32.shape[:2]
f32[h-28:h] = f32[h-28:h] * 0.55 + np.ones((28, w, 3)) * 255 * 0.45
final = f32.astype(np.uint8)

show(original, darkened, final,
     titles=["Original", "Darkened (bad exposure)", "Fixed by pipeline"])
print("\nNumPy operations used in this pipeline:")
print("  • np.clip          → prevent overflow")
print("  • .astype()        → safe dtype conversion")
print("  • np.histogram     → measure pixel distribution")
print("  • cumsum + lut[]   → histogram equalisation (fancy indexing)")
print("  • broadcasting     → watermark alpha blend")


---
## 📚 Quick reference — NumPy operations we used

| Operation | Syntax | What it did |
|-----------|--------|-------------|
| Shape inspection | `img.shape`, `img.dtype` | Understand the array |
| Pixel access | `img[y, x]` | Single pixel RGB |
| Crop | `img[y1:y2, x1:x2]` | Region of interest |
| Channel split | `img[:,:,0]` | Red / Green / Blue |
| Flip | `img[:,::-1]` | Mirror image |
| Vectorised math | `img * 1.5` | Brightness (broadcasting) |
| Safe clipping | `np.clip(arr, 0, 255)` | Prevent overflow |
| Greyscale | `img @ weights` | Weighted channel sum |
| Histogram | `np.histogram(arr, bins=256)` | Pixel distribution |
| Cumulative sum | `arr.cumsum()` | CDF for equalisation |
| Boolean mask | `arr > 150` | Select pixels by value |
| np.where | `np.where(mask, a, b)` | Conditional pixel swap |
| Alpha blend | `(1-α)*img + α*overlay` | Watermark compositing |

---
*Next up: Pandas — using the same array thinking on tabular data.*
